In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/brain-tumor-mri-dataset


In [ ]:
cd /kaggle/input/brain-tumor-mri-dataset

/kaggle/input/brain-tumor-mri-dataset


In [ ]:
ls

Testing/  Training/


In [ ]:

import os

def walk_through(dir_path):
  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")

In [ ]:

walk_through(path)

There are 2 directories and 0 images in '/kaggle/input/brain-tumor-mri-dataset'.
There are 4 directories and 0 images in '/kaggle/input/brain-tumor-mri-dataset/Training'.
There are 0 directories and 1457 images in '/kaggle/input/brain-tumor-mri-dataset/Training/pituitary'.
There are 0 directories and 1595 images in '/kaggle/input/brain-tumor-mri-dataset/Training/notumor'.
There are 0 directories and 1339 images in '/kaggle/input/brain-tumor-mri-dataset/Training/meningioma'.
There are 0 directories and 1321 images in '/kaggle/input/brain-tumor-mri-dataset/Training/glioma'.
There are 4 directories and 0 images in '/kaggle/input/brain-tumor-mri-dataset/Testing'.
There are 0 directories and 300 images in '/kaggle/input/brain-tumor-mri-dataset/Testing/pituitary'.
There are 0 directories and 405 images in '/kaggle/input/brain-tumor-mri-dataset/Testing/notumor'.
There are 0 directories and 306 images in '/kaggle/input/brain-tumor-mri-dataset/Testing/meningioma'.
There are 0 directories and 30

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import os

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
DATA_DIR = '/kaggle/input/brain-tumor-mri-dataset/Training'
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
# Define transformations for the images
# Pre-trained models expect specific normalization
data_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(), # Converts images to PyTorch tensors and scales to [0, 1]
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# --- 2. Load Data and Create DataLoaders ---
# Load the full dataset
full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)
class_names = full_dataset.classes
print("Class names:", class_names)
print(f"Found {len(class_names)} classes.")

# Split the dataset into training and validation sets (80% train, 20% validation)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

Class names: ['glioma', 'meningioma', 'notumor', 'pituitary']
Found 4 classes.


In [ ]:

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)


for param in model.parameters():
    param.requires_grad = False

# Replace the final classifier layer
num_classes = len(class_names)
# Get the number of input features from the layer we are about to replace
num_in_features = model.classifier[1].in_features
# Create the new layer
model.classifier[1] = nn.Linear(in_features=num_in_features, out_features=num_classes)

In [ ]:

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.001)

In [ ]:
num_epochs = 6
for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    running_train_corrects = 0 # <--- ADDED: Initialize training accuracy counter

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1) # <--- ADDED: Get predictions

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_train_corrects += torch.sum(preds == labels.data) # <--- ADDED: Update corrects

    epoch_loss = running_loss / len(train_dataset)
    epoch_train_acc = running_train_corrects.double() / len(train_dataset) # <--- ADDED: Calculate train accuracy

    # <--- MODIFIED: Updated print statement to include training accuracy
    print(f"Epoch {epoch+1}/{num_epochs} - Training Loss: {epoch_loss:.4f} - Training Acc: {epoch_train_acc:.4f}", end=" | ")

Epoch 1/6 - Training Loss: 0.6228 - Training Acc: 0.7993 | Epoch 2/6 - Training Loss: 0.3718 - Training Acc: 0.8785 | Epoch 3/6 - Training Loss: 0.3370 - Training Acc: 0.8840 | Epoch 4/6 - Training Loss: 0.3155 - Training Acc: 0.8917 | Epoch 5/6 - Training Loss: 0.2749 - Training Acc: 0.9114 | Epoch 6/6 - Training Loss: 0.2740 - Training Acc: 0.9030 | 

In [ ]:

    # Validation phase
    model.eval()
    running_corrects = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels.data)

    epoch_acc = running_corrects.double() / len(val_dataset)
    print(f"Validation Accuracy: {epoch_acc:.4f}")

print("Training finished!")

Validation Accuracy: 0.9221
Training finished!


In [ ]:
# Define a path to save the model
model_save_path = "/content/brain_tumor_efficientnet_model.pth"

# Save the model's state dictionary
torch.save(model.state_dict(), model_save_path)

print(f"Model saved to {model_save_path}")

Model saved to /content/brain_tumor_efficientnet_model.pth
